# Breast Cancer Classification: Experiment Summary

**Course:** Deep Learning (COE 443) — Istinye University  
**Instructor:** Asst. Prof. Dr. Yigit Bekir Kaya  
**Dataset:** Breast Cancer Wisconsin (Diagnostic)  
**Task:** Binary Classification — Malignant vs Benign Tumor Detection

---

## Tools & Libraries Used

| Category | Library | Purpose |
|----------|--------|----------|
| Data | pandas, numpy | Data manipulation |
| ML | scikit-learn | Logistic Regression, metrics |
| Deep Learning | PyTorch (torch) | Neural networks |
| Visualization | matplotlib, seaborn | Charts and plots |
| Dataset | sklearn.datasets | Breast Cancer Wisconsin |

---

## Project Steps

| Step | Topic | Week |
|------|-------|------|
| 1 | Data Understanding & EDA | - |
| 2 | Logistic Regression Baseline | Week 2 |
| 3 | Plain MLP (Overfitting) | Week 3 |
| 4 | Regularization (L2 + Early Stopping) | Week 4 |
| 5 | Optimization (Optimizer, Init, LR Schedule) | Week 5 |
| 6 | Final Evaluation | - |

---

## Quick Summary

1. Plain MLP overfits on 569-sample dataset
2. Early Stopping achieved best accuracy: 98.25%
3. Small Random initialization failed (62.28% accuracy)
4. Step Decay was the best LR schedule



In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve().parent
SRC_PATH = str(PROJECT_ROOT / "src")
if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
TABLES_DIR = OUTPUTS_DIR / "tables"
FIGURES_DIR = OUTPUTS_DIR / "figures"

def display_table(csv_path, title):
    df = pd.read_csv(csv_path)
    display(Markdown(f"#### {title}"))
    
    # Format numbers
    df_display = df.copy()
    for col in df_display.columns:
        if df_display[col].dtype in ['float64', 'float32']:
            df_display[col] = df_display[col].apply(lambda x: f"{x:.2f}" if pd.notna(x) else "-")
    
    # Style
    styled = df_display.style.set_properties(**{
        'text-align': 'center', 'font-size': '11px',
    }).set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#f0f0f0'), ('font-weight', 'bold'), ('text-align', 'center')]},
        {'selector': 'td', 'props': [('text-align', 'center')]},
    ])
    display(styled)
    print()

---

# Step 1: Data Understanding

## What We Did

- Loaded Breast Cancer Wisconsin dataset
- Explored 569 samples with 30 features
- Created stratified train/val/test splits (60/20/20)
- Applied StandardScaler (fitted only on training data)

## Key Findings

- 63% Benign, 37% Malignant (class imbalance)
- Features have multicollinearity (up to 0.99 correlation)
- Feature correlations with target range: 0.01 - 0.82



In [ ]:
display_table(f"{TABLES_DIR}/dataset_summary.csv", "Dataset Summary")

In [ ]:
display_table(f"{TABLES_DIR}/split_summary.csv", "Data Split")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

img1 = plt.imread(f"{FIGURES_DIR}/step1/top_feature_correlation_heatmap.png")
img2 = plt.imread(f"{FIGURES_DIR}/step1/class_distribution.png")
img3 = plt.imread(f"{FIGURES_DIR}/step1/top_features_distribution.png")

axes[0].imshow(img1)
axes[0].axis('off')
axes[0].set_title('Correlation Heatmap', fontsize=12)

axes[1].imshow(img2)
axes[1].axis('off')
axes[1].set_title('Class Distribution', fontsize=12)

axes[2].imshow(img3)
axes[2].axis('off')
axes[2].set_title('Top Features Distribution', fontsize=12)

plt.tight_layout()
plt.show()

---

# Step 2: Logistic Regression (Baseline)

## What We Did

- Simple logistic regression (no hidden layers)
- sklearn LogisticRegression with solver='lbfgs'
- Evaluated on validation set

## Why Baseline?

- Represents simplest linear decision boundary
- Performance benchmark for complex models
- If MLP can't beat this, something is wrong



In [ ]:
display_table(f"{TABLES_DIR}/step2_logreg_metrics.csv", "Logistic Regression Results")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

img1 = plt.imread(f"{FIGURES_DIR}/step2/logreg_confusion_matrix.png")
img2 = plt.imread(f"{FIGURES_DIR}/step2/linear_separability_check.png")

axes[0].imshow(img1)
axes[0].axis('off')
axes[0].set_title('Confusion Matrix', fontsize=12)

axes[1].imshow(img2)
axes[1].axis('off')
axes[1].set_title('Linear Separability Check', fontsize=12)

plt.tight_layout()
plt.show()

---

# Step 3: Plain MLP (Overfitting)

## What We Did

- Architecture: 30 → 64 → 32 → 1 (ReLU + Sigmoid)
- ~2,097 parameters
- Trained 200 epochs with Adam optimizer
- No regularization

## Why Overfitting Occurred

| Problem | Effect |
|---------|--------|
| Small dataset (569) | Only 341 training samples |
| Multicollinearity | Feature correlation up to 0.99 |
| Large model | 2,097 params for 341 samples |
| No regularization | Model memorized training data |



In [ ]:
display_table(f"{TABLES_DIR}/step3_combined_metrics.csv", "Step 3: Plain MLP vs Baseline")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

img1 = plt.imread(f"{FIGURES_DIR}/step3/plain_mlp_loss_curves.png")
img2 = plt.imread(f"{FIGURES_DIR}/step3/plain_mlp_confusion_matrix.png")

axes[0].imshow(img1)
axes[0].axis('off')
axes[0].set_title('Loss Curves (Overfitting)', fontsize=12)

axes[1].imshow(img2)
axes[1].axis('off')
axes[1].set_title('Confusion Matrix', fontsize=12)

plt.tight_layout()
plt.show()

---

# Step 4: Regularization

## What We Did

Two methods to fix overfitting:

1. **L2 Regularization:** Added weight penalty (alpha=0.1)
2. **Early Stopping:** Stopped when validation loss stopped improving (patience=10)

## Results

- L2: Recovered baseline performance
- Early Stopping: Best result! 98.25% accuracy
- Stopped at epoch 43 (before memorization)



In [ ]:
display_table(f"{TABLES_DIR}/step4_combined_metrics.csv", "Step 4: Regularization Results")

In [ ]:
img = plt.imread(f"{FIGURES_DIR}/step4/regularized_loss_curves.png")
plt.figure(figsize=(8, 5))
plt.imshow(img)
plt.axis('off')
plt.title('Regularized Loss Curves', fontsize=12)
plt.tight_layout()
plt.show()

---

# Step 5: Optimization

Three experiments (changing one thing at a time):

## Experiment 1: Optimizer Comparison

| Optimizer | Description |
|-----------|-------------|
| Adam | Adaptive per-parameter learning rate |
| SGD + Momentum | Single learning rate with momentum |
| Adam + Cosine | Adam with learning rate schedule |



In [ ]:
display_table(f"{TABLES_DIR}/step5_optimizer_comparison.csv", "Optimizer Comparison")

In [ ]:
img = plt.imread(f"{FIGURES_DIR}/step5/optimizer_comparison.png")
plt.figure(figsize=(8, 5))
plt.imshow(img)
plt.axis('off')
plt.title('Optimizer Comparison', fontsize=12)
plt.tight_layout()
plt.show()

## Experiment 2: Initialization Comparison

| Initialization | Description |
|---------------|-------------|
| He/Kaiming | Designed for ReLU networks |
| Xavier/Glorot | Designed for tanh/sigmoid |
| Small Random (sigma=0.01) | Too small - vanishing gradients |
| Large Random (sigma=1.0) | Too large - unstable |

**Finding:** Small Random initialization **failed catastrophically**!



In [ ]:
display_table(f"{TABLES_DIR}/step5_initialization_comparison.csv", "Initialization Comparison")

In [ ]:
img = plt.imread(f"{FIGURES_DIR}/step5/initialization_comparison.png")
plt.figure(figsize=(8, 5))
plt.imshow(img)
plt.axis('off')
plt.title('Initialization Comparison', fontsize=12)
plt.tight_layout()
plt.show()

## Experiment 3: Learning Rate Schedule

| Schedule | Description |
|----------|-------------|
| Constant LR | No decay |
| Step Decay | Halve LR every 30 epochs |
| Cosine Annealing | Smooth LR decay to zero |

**Finding:** Step Decay achieved highest accuracy!



In [ ]:
display_table(f"{TABLES_DIR}/step5_lr_schedule_comparison.csv", "LR Schedule Comparison")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

img1 = plt.imread(f"{FIGURES_DIR}/step5/lr_schedule_comparison.png")
img2 = plt.imread(f"{FIGURES_DIR}/step5/lr_schedules_visualization.png")

axes[0].imshow(img1)
axes[0].axis('off')
axes[0].set_title('LR Schedule Comparison', fontsize=12)

axes[1].imshow(img2)
axes[1].axis('off')
axes[1].set_title('LR Schedules', fontsize=12)

plt.tight_layout()
plt.show()

---

# Step 6: Final Comparison

All models compared:



In [ ]:
display_table(f"{TABLES_DIR}/step6_master_comparison.csv", "All Models Compared")

In [ ]:
img = plt.imread(f"{FIGURES_DIR}/step5/roc_curve_comparison.png")
plt.figure(figsize=(8, 5))
plt.imshow(img)
plt.axis('off')
plt.title('ROC Curve Comparison', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
img = plt.imread(f"{FIGURES_DIR}/step5/final_metric_comparisons.png")
plt.figure(figsize=(10, 5))
plt.imshow(img)
plt.axis('off')
plt.title('Final Metric Comparisons', fontsize=12)
plt.tight_layout()
plt.show()

---

# Conclusions

## Final Rankings

| Rank | Model | Accuracy |
|------|-------|----------|
| 1 | MLP + Early Stopping | 98.25% |
| 2 | Plain MLP / Step Decay | 97.37% |
| 3 | Logistic Regression | 96.49% |

## Key Takeaways

1. **Early Stopping is best** - Stopped at epoch 43, achieved 98.25% accuracy
2. **Small Random init fails** - 62.28% accuracy due to vanishing gradients
3. **Step Decay works** - Best LR schedule for this dataset
4. **Linear baseline is strong** - 96.49% shows nearly linear separability

## Recommended Configuration

```
Architecture: 30 → 64 → 32 → 1
Optimizer: Adam (lr=0.001)
Initialization: He/Kaiming
LR Schedule: Step Decay
Regularization: Early Stopping
Expected Accuracy: ~98%
```

---

*Project for Deep Learning (COE 443) — Istinye University*